# CliDeck: качество знаний и эксперимент с 20 неизвестными вопросами

30 сентября 2026. Время — UTC. Этот notebook пересчитывает сохранённые доказательства; он не обращается к production и не ставит новые задачи. Открывайте его из каталога отчёта.

Проверяется пополнение базы, не обучение весов модели. Вопросы случайно выбраны с seed 20260930 **после отбора неизвестных**, поэтому 0/20 — результат устранения пробелов за 42,4–43,9 минуты (и на промежуточном срезе 21,7–23,3), а не точность на случайном потоке пользователей. Формальный complete не равен правильному ответу; 15 задач потеряны из-за установленного триггера, один внутренний complete оказался OSPF вместо EVPN. Оставшиеся процессы могут завершиться после окна наблюдения.

Исторический спрос: [2026-08-31 15:50:49Z, 2026-09-30 15:50:49Z). Все 20 тестовых запросов исключены. Органических пользователей нельзя отделить от smoke-тестов по имеющимся анонимным логам. Исторические Fidelity-проверки были до 7 сентября и не оценивают текущую модель.

In [1]:
from analyze import derive, sample_checks
import json
metrics, rows = derive()
print(json.dumps(metrics, ensure_ascii=False, indent=2))


{
  "experiment": {
    "seed": 20260930,
    "questions": 20,
    "groups": {
      "linux": 13,
      "openwrt": 4,
      "junos": 1,
      "arista": 1,
      "cisco": 1
    },
    "recorded_requests": 20,
    "completed_diagnoses": 5,
    "without_diagnosis_task": 15,
    "correct_new_answers_observed": 0,
    "final_statuses": {
      "unknown": 20
    },
    "observation_min_minutes": 21.69,
    "observation_max_minutes": 23.275,
    "demand_states": {
      "processing": 3,
      "diagnosing": 15,
      "published": 1,
      "discovering": 1
    },
    "tasks": 66,
    "task_states": {
      "completed": 64,
      "running": 1,
      "queued": 1
    },
    "agent_runs": 61,
    "agent_run_states": {
      "completed": 58,
      "failed": 2,
      "running": 1
    },
    "sources_linked": 13,
    "sources_newly_discovered": 1,
    "sources_reused": 12
  },
  "quality": {
    "active": 351110,
    "public": 351094,
    "no_provenance": 56747,
    "no_provenance_pct": 16.1622,
    "

## Каждый вопрос и время наблюдения

Время diagnosis считается от durable first_seen_at до completed_at задачи. Последняя проверка использует исходный контекст, без постановки новых задач. Статусы задач сняты в 16:14:02Z; ответы проверены между 16:14:05Z и 16:14:53Z.

## Заключительная проверка
16:34:35Z — процесс; 16:34:40–16:35:38Z — ответы. Никаких repair/config действий между контрольными срезами.


In [2]:
closing_metrics, closing_rows = derive("monitor-closing.json", "recheck-closing.json")
print(json.dumps(closing_metrics["experiment"], ensure_ascii=False, indent=2))
for row in closing_rows:
    print(f"{row['experiment_id']:02d}: {row['observation_minutes']:.2f} min, {row['demand_status']}, {row['final_answer_status']}")


{
  "seed": 20260930,
  "questions": 20,
  "groups": {
    "linux": 13,
    "openwrt": 4,
    "junos": 1,
    "arista": 1,
    "cisco": 1
  },
  "recorded_requests": 20,
  "completed_diagnoses": 5,
  "without_diagnosis_task": 15,
  "correct_new_answers_observed": 0,
  "final_statuses": {
    "unknown": 20
  },
  "observation_min_minutes": 42.427,
  "observation_max_minutes": 43.855,
  "demand_states": {
    "processing": 3,
    "diagnosing": 15,
    "published": 1,
    "acquiring": 1
  },
  "tasks": 118,
  "task_states": {
    "completed": 114,
    "failed": 2,
    "running": 2
  },
  "agent_runs": 109,
  "agent_run_states": {
    "completed": 102,
    "failed": 5,
    "running": 2
  },
  "sources_linked": 15,
  "sources_newly_discovered": 3,
  "sources_reused": 12
}
01: 43.85 min, processing, unknown
02: 42.75 min, processing, unknown
03: 42.76 min, diagnosing, unknown
04: 42.76 min, diagnosing, unknown
05: 42.75 min, diagnosing, unknown
06: 42.74 min, diagnosing, unknown
07: 42.73 mi

In [3]:
for row in rows:
    print(f"{row['experiment_id']:02d} | {row['observation_minutes']:.2f} min | diagnosis={row['diagnosis_seconds']} s | {row['demand_status']} | final={row['final_answer_status']} | {row['question']}")


01 | 23.27 min | diagnosis=13.102 s | processing | final=unknown | How do I display the Linux netfilter conntrack expectation table without deleting entries?
02 | 22.16 min | diagnosis=10.923 s | processing | final=unknown | How can I inspect a Linux MACsec interface cipher suite and replay protection settings?
03 | 22.14 min | diagnosis=None s | diagnosing | final=unknown | How can I identify packet drops caused by XDP redirect failures using Linux tracepoints?
04 | 22.12 min | diagnosis=None s | diagnosing | final=unknown | How can I inspect an SRv6 seg6local route and its End.DT6 behavior with iproute2?
05 | 22.12 min | diagnosis=None s | diagnosing | final=unknown | How do I diagnose DNS over HTTPS bootstrap lookup failures in an OpenWrt https-dns-proxy deployment?
06 | 22.10 min | diagnosis=None s | diagnosing | final=unknown | How can I inspect Junos SRv6 locator and local SID allocation state?
07 | 22.07 min | diagnosis=10.529 s | published | final=unknown | How can I inspect Ar

## Ручная проверка случайных записей

30 активных researcher-записей выбраны ORDER BY md5(public_ref || seed), без legacy_import. Проверены описания и привязанные окна извлечения; это не лабораторное исполнение всех команд. Поиск буквального совпадения после нормализации пробелов полезен для диагностики цитат, но не измеряет фактическую точность: placeholders, OCR и парафразы меняют строки.

In [4]:
checks = sample_checks()
for row in checks:
    if row["manual_defect"]:
        print(json.dumps({key:row[key] for key in ("sample_number", "public_ref", "command_text", "manual_defect")}, ensure_ascii=False))
print("Evidence excerpts not literally in extraction window:", sum(row["evidence_literal_in_window"] is False for row in checks), "/", len(checks))


{"sample_number": 4, "public_ref": "6ae5d3b1-4ee3-46e1-891c-3cef020a3f97", "command_text": "no boot config-file", "manual_defect": "Описание про stacking Catalyst 2960-S не объясняет no boot config-file."}
{"sample_number": 9, "public_ref": "9526c416-185c-4988-8a9c-15a61a4c09af", "command_text": "no spanning-tree mst pre-standard", "manual_defect": "Описание обрезано до 'form of this command.' и не объясняет команду."}
{"sample_number": 23, "public_ref": "18dce091-1286-4f25-9d65-6e01cb794578", "command_text": "[Page 3339] Commands Syntax Description", "manual_defect": "Заголовок страницы и таблицы опубликован как команда; окно содержит show switch bluetooth."}
Evidence excerpts not literally in extraction window: 15 / 30


## Выполненные SQL и источники

baseline.json содержит согласованный структурный срез 15:51:56Z. Исторические usage-секции этого файла не являются финальным знаменателем: уточнённое окно и финальные числа сохранены отдельно в usage-window.sql/json. Deep-quality и последующие запросы — последовательные READ ONLY срезы работающей базы; различие 351110/351120 отражает поступление записей между срезами. audit-extra.json — подтверждение триггера и нерелевантного ответа в 16:23:29Z. Точные результаты и 20 HTTP-ответов сохранены рядом.

### baseline.sql

```sql
BEGIN ISOLATION LEVEL REPEATABLE READ READ ONLY;
SET LOCAL statement_timeout='45s';
SELECT json_build_object('section','snapshot','at',now(),'database',current_database(),'active_release',(SELECT row_to_json(t) FROM (SELECT ar.release_id,r.sequence,r.created_at FROM active_release ar LEFT JOIN releases r ON r.id=ar.release_id) t),'stats_cache',(SELECT row_to_json(t) FROM (SELECT refreshed_at,refresh_error_code,payload->>'generated_at' generated_at,payload->'coverage'->>'published_knowledge' reported_knowledge FROM public_stats_cache) t),'pipeline',(SELECT row_to_json(t) FROM (SELECT enabled,max_concurrent_ai_runs,settings_version,paused_reason FROM pipeline_settings) t),'profiles',(SELECT json_agg(row_to_json(t)) FROM pipeline_execution_profiles t));
SELECT json_build_object('section','volume','active',(SELECT count(*) FROM active_knowledge_state),'public_rows',(SELECT count(*) FROM public_active_knowledge),'public_distinct_revisions',(SELECT count(DISTINCT revision_id) FROM public_active_knowledge),'items',(SELECT count(*) FROM knowledge_items),'revisions',(SELECT count(*) FROM knowledge_revisions),'candidates',(SELECT count(*) FROM knowledge_candidates),'sources',(SELECT count(*) FROM source_candidates),'fragments',(SELECT count(*) FROM source_fragments),'last_revision',(SELECT max(created_at) FROM knowledge_revisions),'last_release',(SELECT max(created_at) FROM releases WHERE status='published'));
SELECT json_build_object('section','public_quality','metrics',row_to_json(t)) FROM (SELECT count(*) records,count(*) FILTER(WHERE title='' OR summary='') empty_text,count(*) FILTER(WHERE command_text IS NOT NULL) commands,count(*) FILTER(WHERE jsonb_array_length(procedure_steps)>0) procedures,count(*) FILTER(WHERE cardinality(question_patterns)=0) missing_patterns,count(*) FILTER(WHERE jsonb_array_length(verification_steps)=0) missing_verification,count(*) FILTER(WHERE dangerous) dangerous,count(*) FILTER(WHERE dangerous AND jsonb_array_length(verification_steps)=0) dangerous_without_verification,count(*) FILTER(WHERE dangerous AND jsonb_array_length(rollback_steps)=0) dangerous_without_rollback,count(*) FILTER(WHERE risk_level='unknown') risk_unknown,count(*) FILTER(WHERE validation_level='runtime_lab_validated') lab_validated,count(*) FILTER(WHERE validation_level='documentation_reviewed') documentation_reviewed,count(*) FILTER(WHERE next_review_at < now()) overdue,count(*) FILTER(WHERE operating_system_slug IS NULL) no_os,count(*) FILTER(WHERE vendor_slug IS NULL) no_vendor,count(*) FILTER(WHERE last_verified_at > CURRENT_DATE) future_verified,percentile_cont(ARRAY[0.1,0.5,0.9]) WITHIN GROUP(ORDER BY confidence::float) confidence_quantiles FROM public_active_knowledge) t;
SELECT json_build_object('section','provenance','metrics',row_to_json(t)) FROM (SELECT count(*) active,count(*) FILTER(WHERE NOT EXISTS(SELECT 1 FROM revision_sources rs WHERE rs.revision_id=aks.revision_id)) no_source,count(*) FILTER(WHERE NOT EXISTS(SELECT 1 FROM revision_sources rs JOIN source_documents sd ON sd.id=rs.source_document_id WHERE rs.revision_id=aks.revision_id AND rs.evidence_role='primary' AND length(sd.evidence_fragment)>0 AND sd.content_hash ~ '^sha256:[0-9a-f]{64}$')) no_valid_primary,count(*) FILTER(WHERE NOT EXISTS(SELECT 1 FROM knowledge_applicability_index ai WHERE ai.revision_id=aks.revision_id)) no_applicability FROM active_knowledge_state aks) t;
SELECT json_build_object('section','duplicates','groups',count(*),'rows_in_groups',coalesce(sum(n),0),'extra_rows',coalesce(sum(n-1),0)) FROM (SELECT count(*) n FROM public_active_knowledge WHERE command_text IS NOT NULL GROUP BY operating_system_slug,vendor_slug,coalesce(platform_slug,''),coalesce(version_min,''),coalesce(version_max,''),regexp_replace(lower(trim(command_text)),'\s+',' ','g') HAVING count(*)>1) t;
SELECT json_build_object('section','os_distribution','rows',json_agg(row_to_json(t))) FROM (SELECT operating_system_slug,count(*) n FROM public_active_knowledge GROUP BY 1 ORDER BY 2 DESC LIMIT 35) t;
SELECT json_build_object('section','source_status','rows',json_agg(row_to_json(t))) FROM (SELECT status,source_kind,failure_code,count(*) n FROM source_candidates GROUP BY 1,2,3 ORDER BY 4 DESC LIMIT 35) t;
SELECT json_build_object('section','run_status','rows',json_agg(row_to_json(t))) FROM (SELECT status,failure_code,count(*) n FROM source_processing_runs GROUP BY 1,2 ORDER BY 3 DESC LIMIT 25) t;
SELECT json_build_object('section','fragments','rows',json_agg(row_to_json(t))) FROM (SELECT status,disposition,disposition_reason,count(*) n FROM source_fragments GROUP BY 1,2,3 ORDER BY 4 DESC) t;
SELECT json_build_object('section','candidate_status','rows',json_agg(row_to_json(t))) FROM (SELECT status,fidelity_status,count(*) n FROM knowledge_candidates GROUP BY 1,2 ORDER BY 3 DESC LIMIT 30) t;
SELECT json_build_object('section','qa','rows',json_agg(row_to_json(t))) FROM (SELECT stage,status,material_error,count(*) n,sum(coverage_count) coverage,sum(input_tokens) input_tokens,sum(output_tokens) output_tokens,min(created_at) first_at,max(created_at) last_at FROM pipeline_quality_checks GROUP BY 1,2,3 ORDER BY 4 DESC) t;
SELECT json_build_object('section','agent_24h','rows',json_agg(row_to_json(t))) FROM (SELECT model,reasoning_effort,status,error_code,count(*) n,round(avg(duration_ms)::numeric) avg_ms,sum(input_tokens) input_tokens,sum(cached_input_tokens) cached_input_tokens,sum(output_tokens) output_tokens FROM agent_runs WHERE started_at > now()-interval '24 hours' GROUP BY 1,2,3,4 ORDER BY 5 DESC) t;
SELECT json_build_object('section','agent_days','rows',json_agg(row_to_json(t))) FROM (SELECT (started_at AT TIME ZONE 'America/Chicago')::date AS day,status,count(*) n,sum(input_tokens) input_tokens,sum(output_tokens) output_tokens FROM agent_runs WHERE started_at > now()-interval '14 days' GROUP BY 1,2 ORDER BY 1,2) t;
SELECT json_build_object('section','demands','rows',json_agg(row_to_json(t))) FROM (SELECT status,diagnosis_status,replay_status,last_error_code,count(*) n,sum(demand_count) requests,min(first_seen_at) first_seen,max(last_seen_at) last_seen FROM knowledge_demands GROUP BY 1,2,3,4 ORDER BY 5 DESC LIMIT 35) t;
SELECT json_build_object('section','usage30d','rows',json_agg(row_to_json(t))) FROM (SELECT tool_name,outcome,response_payload->>'answer_status' answer_status,actor_kind,count(*) n,count(DISTINCT question_preview) unique_questions,count(*) FILTER(WHERE knowledge_demand_id IS NOT NULL) linked_demands,round(avg(duration_ms)::numeric) avg_ms FROM mcp_request_logs WHERE occurred_at > '2026-08-31 15:50:49+00' AND occurred_at < '2026-09-30 15:50:49+00' GROUP BY 1,2,3,4 ORDER BY 5 DESC) t;
SELECT json_build_object('section','usage_daily30d','rows',json_agg(row_to_json(t))) FROM (SELECT day,operation,outcome,sum(request_count) n FROM public_usage_daily WHERE day >= CURRENT_DATE-30 GROUP BY 1,2,3 ORDER BY 1,2,3) t;
SELECT json_build_object('section','revisions_daily14d','rows',json_agg(row_to_json(t))) FROM (SELECT (created_at AT TIME ZONE 'America/Chicago')::date AS day,created_by,count(*) n FROM knowledge_revisions WHERE created_at > now()-interval '14 days' GROUP BY 1,2 ORDER BY 1,2) t;
COMMIT;

```

### deep-quality.sql

```sql
BEGIN READ ONLY;
SET LOCAL statement_timeout='45s';
SELECT json_build_object('section','missing_provenance_segments','rows',json_agg(row_to_json(t))) FROM (SELECT kr.created_by,kr.domain_id,min(kr.created_at) first_at,max(kr.created_at) last_at,count(*) n,count(*) FILTER(WHERE kr.created_at>now()-interval '30 days') recent FROM active_knowledge_state aks JOIN knowledge_revisions kr ON kr.id=aks.revision_id WHERE NOT EXISTS(SELECT 1 FROM revision_sources rs WHERE rs.revision_id=aks.revision_id) GROUP BY 1,2) t;
WITH quality AS MATERIALIZED (SELECT revision_id,bool_or(fidelity_status='passed') passed,bool_or(fidelity_status='repair') repair,bool_or(fidelity_status='unavailable') unavailable,bool_or(fidelity_status='checking') checking FROM knowledge_candidates WHERE revision_id IS NOT NULL GROUP BY revision_id) SELECT json_build_object('section','active_fidelity','metrics',row_to_json(t)) FROM (SELECT count(*) active,count(q.revision_id) has_candidate,count(*) FILTER(WHERE q.passed) passed,count(*) FILTER(WHERE q.repair) repair,count(*) FILTER(WHERE q.unavailable) unavailable,count(*) FILTER(WHERE q.checking) checking FROM active_knowledge_state aks LEFT JOIN quality q ON q.revision_id=aks.revision_id) t;
SELECT json_build_object('section','qa_24h','rows',json_agg(row_to_json(t))) FROM (SELECT stage,status,material_error,count(*) n,max(created_at) latest FROM pipeline_quality_checks WHERE created_at>now()-interval '24 hours' GROUP BY 1,2,3) t;
SELECT json_build_object('section','dangling_diagnoses_baseline','metrics',row_to_json(t)) FROM (SELECT count(*) diagnosing,count(*) FILTER(WHERE diagnosis_task_id IS NULL) no_task_pointer,count(*) FILTER(WHERE NOT EXISTS(SELECT 1 FROM pipeline_tasks pt WHERE pt.knowledge_demand_id=kd.id AND pt.task_type='demand_diagnosis' AND pt.status IN ('queued','claimed','running'))) no_live_task,min(first_seen_at) oldest FROM knowledge_demands kd WHERE kd.status='diagnosing' AND kd.first_seen_at<'2026-09-30 15:50:49+00') t;
SELECT json_build_object('section','demand_before_experiment','metrics',row_to_json(t)) FROM (SELECT count(*) total,count(*) FILTER(WHERE status='published') published,count(*) FILTER(WHERE replay_status='complete') replay_complete,count(*) FILTER(WHERE status='unresolved') unresolved,count(*) FILTER(WHERE replay_status='partial') partial,min(first_seen_at) first_seen,max(first_seen_at) last_seen FROM knowledge_demands WHERE first_seen_at<'2026-09-30 15:50:49+00') t;
SELECT json_build_object('section','served_revisions30d','metrics',row_to_json(t)) FROM (SELECT count(*) references,count(DISTINCT answer->>'revision_ref') distinct_revisions FROM mcp_request_logs ml CROSS JOIN LATERAL jsonb_array_elements(CASE WHEN jsonb_typeof(ml.response_payload->'answers')='array' THEN ml.response_payload->'answers' ELSE '[]'::jsonb END) answer WHERE ml.occurred_at>'2026-08-31 15:50:49+00' AND ml.occurred_at<'2026-09-30 15:50:49+00' AND ml.tool_name IN ('query_network_knowledge','get_network_workflow','query_domain_knowledge')) t;
SELECT json_build_object('section','snapshot_traffic','rows',json_agg(row_to_json(t))) FROM (SELECT (occurred_at AT TIME ZONE 'America/Chicago')::date AS day,tool_name,count(*) n FROM mcp_request_logs WHERE occurred_at > now()-interval '14 days' AND occurred_at<'2026-09-30 15:50:49+00' AND tool_name IN ('query_network_knowledge','get_network_workflow','query_domain_knowledge') GROUP BY 1,2 ORDER BY 1,2) t;
SELECT json_build_object('section','quality_by_origin','rows',json_agg(row_to_json(t))) FROM (SELECT kr.created_by,count(*) n,count(*) FILTER(WHERE kr.risk_level='unknown') risk_unknown,count(*) FILTER(WHERE kr.dangerous) dangerous,count(*) FILTER(WHERE kr.dangerous AND jsonb_array_length(kr.verification_steps)=0) dangerous_without_verification,count(*) FILTER(WHERE kr.dangerous AND jsonb_array_length(kr.rollback_steps)=0) dangerous_without_rollback,count(*) FILTER(WHERE kr.last_verified_at>CURRENT_DATE) future_verified FROM active_knowledge_state aks JOIN knowledge_revisions kr ON kr.id=aks.revision_id GROUP BY 1) t;
WITH sample AS MATERIALIZED (SELECT kr.id,kr.public_ref FROM active_knowledge_state aks JOIN knowledge_revisions kr ON kr.id=aks.revision_id WHERE kr.created_by='researcher' ORDER BY md5(kr.public_ref::text||'20260930') LIMIT 30) SELECT json_build_object('section','source_fidelity_sample','rows',json_agg(row_to_json(t))) FROM (SELECT kr.public_ref,kr.title,kr.summary,kr.command_text,kr.procedure_steps,kr.risk_level,sd.canonical_url,sd.content_hash,sd.evidence_fragment,rs.evidence_role FROM sample s JOIN knowledge_revisions kr ON kr.id=s.id LEFT JOIN revision_sources rs ON rs.revision_id=kr.id AND rs.evidence_role='primary' LEFT JOIN source_documents sd ON sd.id=rs.source_document_id) t;
COMMIT;

```

### usage-window.sql

```sql
BEGIN READ ONLY;
SET LOCAL statement_timeout='30s';
SELECT json_build_object('section','network_questions','at',now(),'metrics',row_to_json(t)) FROM (SELECT count(*) calls,count(DISTINCT question_preview) questions,count(*) FILTER(WHERE response_payload->>'answer_status'='complete') complete,count(*) FILTER(WHERE response_payload->>'answer_status'='partial') partial,count(*) FILTER(WHERE response_payload->>'answer_status'='unknown') unknown FROM mcp_request_logs WHERE tool_name='query_network_knowledge' AND occurred_at >= '2026-08-31 15:50:49+00' AND occurred_at < '2026-09-30 15:50:49+00') t;
WITH served AS (SELECT DISTINCT answer->>'revision_ref' ref FROM mcp_request_logs ml CROSS JOIN LATERAL jsonb_array_elements(CASE WHEN jsonb_typeof(ml.response_payload->'answers')='array' THEN ml.response_payload->'answers' ELSE '[]'::jsonb END) answer WHERE ml.occurred_at>='2026-08-31 15:50:49+00' AND ml.occurred_at<'2026-09-30 15:50:49+00' AND ml.tool_name IN ('query_network_knowledge','get_network_workflow','query_domain_knowledge')) SELECT json_build_object('section','served_active_items','at',now(),'metrics',row_to_json(t)) FROM (SELECT count(DISTINCT kr.knowledge_item_id) active_items_served,count(DISTINCT kr.id) historical_revisions_served FROM served s JOIN knowledge_revisions kr ON kr.public_ref::text=s.ref JOIN active_knowledge_state aks ON aks.knowledge_item_id=kr.knowledge_item_id) t;
SELECT json_build_object('section','smoke_fixture_matches','at',now(),'metrics',row_to_json(t)) FROM (SELECT count(*) calls_matching_syslog_smoke FROM mcp_request_logs WHERE tool_name='query_network_knowledge' AND lower(question_preview)='configure remote syslog server' AND occurred_at >= '2026-08-31 15:50:49+00' AND occurred_at < '2026-09-30 15:50:49+00') t;
COMMIT;

```

### monitor.sql

```sql
BEGIN READ ONLY;
SET LOCAL statement_timeout='25s';
SELECT json_build_object('section','demands','at',now(),'rows',json_agg(row_to_json(t))) FROM (SELECT id,question,status,diagnosis_status,replay_status,last_error_code,demand_count,first_seen_at,last_seen_at,next_retry_at,completed_at,result_revision_id FROM knowledge_demands WHERE question IN ('How do I display the Linux netfilter conntrack expectation table without deleting entries?','How can I inspect a Linux MACsec interface cipher suite and replay protection settings?','How can I identify packet drops caused by XDP redirect failures using Linux tracepoints?','How can I inspect an SRv6 seg6local route and its End.DT6 behavior with iproute2?','How do I diagnose DNS over HTTPS bootstrap lookup failures in an OpenWrt https-dns-proxy deployment?','How can I inspect Junos SRv6 locator and local SID allocation state?','How can I inspect Arista EOS EVPN multihoming designated forwarder election state?','How do I identify whether Linux reverse path filtering uses strict or loose mode on an interface?','How can I inspect Linux network device RSS hash functions and indirection tables?','How can I inspect a tc cake qdisc flow isolation mode and packet drop counters?','How do I inspect the per-network-namespace nftables flowtable offload configuration?','How can I inspect Multipath TCP endpoint flags and connection subflows with ip mptcp?','How can I compare Linux IPv6 route cache exceptions with the main IPv6 routing table?','How do I verify OpenWrt mwan3 policy routing rules for an existing connection?','How can I inspect the active OpenWrt SQM cake overhead compensation settings?','How can I inspect Linux bridge multicast database entries and their source-specific filters?','How can I inspect OpenWrt firewall4 nftables flow offloading rules without restarting the firewall?','How do I identify whether a Linux UDP socket uses UDP_SEGMENT or UDP_GRO?','How can I inspect ethtool receive flow classification rules without modifying them?','How do I verify IOS XE segment routing microloop avoidance state?') ORDER BY first_seen_at) t;
SELECT json_build_object('section','tasks','at',now(),'rows',json_agg(row_to_json(t))) FROM (SELECT pt.id,pt.knowledge_demand_id,pt.task_type,pt.status,pt.failure_code,pt.created_at,pt.updated_at,pt.completed_at,pt.result FROM pipeline_tasks pt JOIN knowledge_demands kd ON kd.id=pt.knowledge_demand_id WHERE kd.question IN ('How do I display the Linux netfilter conntrack expectation table without deleting entries?','How can I inspect a Linux MACsec interface cipher suite and replay protection settings?','How can I identify packet drops caused by XDP redirect failures using Linux tracepoints?','How can I inspect an SRv6 seg6local route and its End.DT6 behavior with iproute2?','How do I diagnose DNS over HTTPS bootstrap lookup failures in an OpenWrt https-dns-proxy deployment?','How can I inspect Junos SRv6 locator and local SID allocation state?','How can I inspect Arista EOS EVPN multihoming designated forwarder election state?','How do I identify whether Linux reverse path filtering uses strict or loose mode on an interface?','How can I inspect Linux network device RSS hash functions and indirection tables?','How can I inspect a tc cake qdisc flow isolation mode and packet drop counters?','How do I inspect the per-network-namespace nftables flowtable offload configuration?','How can I inspect Multipath TCP endpoint flags and connection subflows with ip mptcp?','How can I compare Linux IPv6 route cache exceptions with the main IPv6 routing table?','How do I verify OpenWrt mwan3 policy routing rules for an existing connection?','How can I inspect the active OpenWrt SQM cake overhead compensation settings?','How can I inspect Linux bridge multicast database entries and their source-specific filters?','How can I inspect OpenWrt firewall4 nftables flow offloading rules without restarting the firewall?','How do I identify whether a Linux UDP socket uses UDP_SEGMENT or UDP_GRO?','How can I inspect ethtool receive flow classification rules without modifying them?','How do I verify IOS XE segment routing microloop avoidance state?') ORDER BY pt.created_at) t;
SELECT json_build_object('section','agent_runs','at',now(),'rows',json_agg(row_to_json(t))) FROM (SELECT ar.id,pt.knowledge_demand_id,pt.task_type,ar.execution_profile,ar.model,ar.reasoning_effort,ar.status,ar.error_code,ar.duration_ms,ar.input_tokens,ar.output_tokens,ar.started_at,ar.completed_at FROM agent_runs ar JOIN pipeline_tasks pt ON pt.id=ar.pipeline_task_id JOIN knowledge_demands kd ON kd.id=pt.knowledge_demand_id WHERE kd.question IN ('How do I display the Linux netfilter conntrack expectation table without deleting entries?','How can I inspect a Linux MACsec interface cipher suite and replay protection settings?','How can I identify packet drops caused by XDP redirect failures using Linux tracepoints?','How can I inspect an SRv6 seg6local route and its End.DT6 behavior with iproute2?','How do I diagnose DNS over HTTPS bootstrap lookup failures in an OpenWrt https-dns-proxy deployment?','How can I inspect Junos SRv6 locator and local SID allocation state?','How can I inspect Arista EOS EVPN multihoming designated forwarder election state?','How do I identify whether Linux reverse path filtering uses strict or loose mode on an interface?','How can I inspect Linux network device RSS hash functions and indirection tables?','How can I inspect a tc cake qdisc flow isolation mode and packet drop counters?','How do I inspect the per-network-namespace nftables flowtable offload configuration?','How can I inspect Multipath TCP endpoint flags and connection subflows with ip mptcp?','How can I compare Linux IPv6 route cache exceptions with the main IPv6 routing table?','How do I verify OpenWrt mwan3 policy routing rules for an existing connection?','How can I inspect the active OpenWrt SQM cake overhead compensation settings?','How can I inspect Linux bridge multicast database entries and their source-specific filters?','How can I inspect OpenWrt firewall4 nftables flow offloading rules without restarting the firewall?','How do I identify whether a Linux UDP socket uses UDP_SEGMENT or UDP_GRO?','How can I inspect ethtool receive flow classification rules without modifying them?','How do I verify IOS XE segment routing microloop avoidance state?') ORDER BY ar.started_at) t;
SELECT json_build_object('section','sources','at',now(),'rows',json_agg(row_to_json(t))) FROM (SELECT sc.id,sc.knowledge_demand_id,sc.canonical_url,sc.status,sc.failure_code,sc.discovered_at,sc.completed_at FROM source_candidates sc JOIN knowledge_demands kd ON kd.id=sc.knowledge_demand_id WHERE kd.question IN ('How do I display the Linux netfilter conntrack expectation table without deleting entries?','How can I inspect a Linux MACsec interface cipher suite and replay protection settings?','How can I identify packet drops caused by XDP redirect failures using Linux tracepoints?','How can I inspect an SRv6 seg6local route and its End.DT6 behavior with iproute2?','How do I diagnose DNS over HTTPS bootstrap lookup failures in an OpenWrt https-dns-proxy deployment?','How can I inspect Junos SRv6 locator and local SID allocation state?','How can I inspect Arista EOS EVPN multihoming designated forwarder election state?','How do I identify whether Linux reverse path filtering uses strict or loose mode on an interface?','How can I inspect Linux network device RSS hash functions and indirection tables?','How can I inspect a tc cake qdisc flow isolation mode and packet drop counters?','How do I inspect the per-network-namespace nftables flowtable offload configuration?','How can I inspect Multipath TCP endpoint flags and connection subflows with ip mptcp?','How can I compare Linux IPv6 route cache exceptions with the main IPv6 routing table?','How do I verify OpenWrt mwan3 policy routing rules for an existing connection?','How can I inspect the active OpenWrt SQM cake overhead compensation settings?','How can I inspect Linux bridge multicast database entries and their source-specific filters?','How can I inspect OpenWrt firewall4 nftables flow offloading rules without restarting the firewall?','How do I identify whether a Linux UDP socket uses UDP_SEGMENT or UDP_GRO?','How can I inspect ethtool receive flow classification rules without modifying them?','How do I verify IOS XE segment routing microloop avoidance state?') ORDER BY sc.discovered_at) t;
SELECT json_build_object('section','requests','at',now(),'rows',json_agg(row_to_json(t))) FROM (SELECT ml.id,ml.tool_name,ml.outcome,ml.response_payload->>'answer_status' answer_status,ml.response_payload->'learning' learning,ml.knowledge_demand_id,ml.duration_ms,ml.error_code,ml.occurred_at FROM mcp_request_logs ml WHERE ml.request_payload->>'question' IN ('How do I display the Linux netfilter conntrack expectation table without deleting entries?','How can I inspect a Linux MACsec interface cipher suite and replay protection settings?','How can I identify packet drops caused by XDP redirect failures using Linux tracepoints?','How can I inspect an SRv6 seg6local route and its End.DT6 behavior with iproute2?','How do I diagnose DNS over HTTPS bootstrap lookup failures in an OpenWrt https-dns-proxy deployment?','How can I inspect Junos SRv6 locator and local SID allocation state?','How can I inspect Arista EOS EVPN multihoming designated forwarder election state?','How do I identify whether Linux reverse path filtering uses strict or loose mode on an interface?','How can I inspect Linux network device RSS hash functions and indirection tables?','How can I inspect a tc cake qdisc flow isolation mode and packet drop counters?','How do I inspect the per-network-namespace nftables flowtable offload configuration?','How can I inspect Multipath TCP endpoint flags and connection subflows with ip mptcp?','How can I compare Linux IPv6 route cache exceptions with the main IPv6 routing table?','How do I verify OpenWrt mwan3 policy routing rules for an existing connection?','How can I inspect the active OpenWrt SQM cake overhead compensation settings?','How can I inspect Linux bridge multicast database entries and their source-specific filters?','How can I inspect OpenWrt firewall4 nftables flow offloading rules without restarting the firewall?','How do I identify whether a Linux UDP socket uses UDP_SEGMENT or UDP_GRO?','How can I inspect ethtool receive flow classification rules without modifying them?','How do I verify IOS XE segment routing microloop avoidance state?') AND ml.occurred_at >= '2026-09-30 15:50:00+00' ORDER BY ml.occurred_at) t;
SELECT json_build_object('section','executors','at',now(),'rows',json_agg(row_to_json(t))) FROM (SELECT worker_name,heartbeat_at,metadata->>'status' status,metadata->>'reason' reason FROM worker_heartbeats WHERE worker_name LIKE 'pipeline-executor-%' ORDER BY worker_name) t;
COMMIT;

```

### audit-extra.sql

```sql
BEGIN READ ONLY;
SET LOCAL statement_timeout='30s';
SELECT json_build_object('section','discovery_payloads','at',now(),'rows',json_agg(row_to_json(t))) FROM (SELECT kd.question,pt.id,pt.status,pt.created_at,pt.payload,pt.result FROM pipeline_tasks pt JOIN knowledge_demands kd ON kd.id=pt.knowledge_demand_id WHERE kd.question IN ('How do I display the Linux netfilter conntrack expectation table without deleting entries?','How can I inspect a Linux MACsec interface cipher suite and replay protection settings?','How can I identify packet drops caused by XDP redirect failures using Linux tracepoints?','How can I inspect an SRv6 seg6local route and its End.DT6 behavior with iproute2?','How do I diagnose DNS over HTTPS bootstrap lookup failures in an OpenWrt https-dns-proxy deployment?','How can I inspect Junos SRv6 locator and local SID allocation state?','How can I inspect Arista EOS EVPN multihoming designated forwarder election state?','How do I identify whether Linux reverse path filtering uses strict or loose mode on an interface?','How can I inspect Linux network device RSS hash functions and indirection tables?','How can I inspect a tc cake qdisc flow isolation mode and packet drop counters?','How do I inspect the per-network-namespace nftables flowtable offload configuration?','How can I inspect Multipath TCP endpoint flags and connection subflows with ip mptcp?','How can I compare Linux IPv6 route cache exceptions with the main IPv6 routing table?','How do I verify OpenWrt mwan3 policy routing rules for an existing connection?','How can I inspect the active OpenWrt SQM cake overhead compensation settings?','How can I inspect Linux bridge multicast database entries and their source-specific filters?','How can I inspect OpenWrt firewall4 nftables flow offloading rules without restarting the firewall?','How do I identify whether a Linux UDP socket uses UDP_SEGMENT or UDP_GRO?','How can I inspect ethtool receive flow classification rules without modifying them?','How do I verify IOS XE segment routing microloop avoidance state?') AND pt.task_type='source_discovery' ORDER BY pt.created_at) t;
SELECT json_build_object('section','published_fidelity_source_states','at',now(),'rows',json_agg(row_to_json(t))) FROM (SELECT sc.status source_status,kc.fidelity_status,count(*) n FROM knowledge_candidates kc JOIN pipeline_tasks pt ON pt.id=kc.pipeline_task_id JOIN source_candidates sc ON sc.id=pt.source_candidate_id WHERE kc.status='published' GROUP BY 1,2 ORDER BY 3 DESC) t;
SELECT json_build_object('section','false_answer_revision','at',now(),'rows',json_agg(row_to_json(t))) FROM (SELECT kr.public_ref,kr.title,kr.summary,kr.command_text,kr.created_at,kp.vendor_slug,kp.operating_system_slug FROM knowledge_revisions kr LEFT JOIN public_active_knowledge kp ON kp.revision_id=kr.id WHERE kr.public_ref='805b01c1-a2de-4f70-baf3-f5b42080d129') t;
SELECT json_build_object('section','live_diagnosis_trigger','at',now(),'rows',json_agg(row_to_json(t))) FROM (SELECT tgname,tgenabled,pg_get_triggerdef(oid) definition,pg_get_functiondef(tgfoid) function_definition FROM pg_trigger WHERE tgname='pipeline_tasks_single_diagnosis') t;
COMMIT;

```

### sample-windows.sql

```sql
BEGIN READ ONLY;
SET LOCAL statement_timeout='30s';
WITH sample AS MATERIALIZED (SELECT id,public_ref FROM knowledge_revisions WHERE public_ref IN ('f0406ebc-25e9-4b96-b352-b094cf9df46b','7f5bd62b-cb20-4ec9-8e53-637512de2f78','36e7c223-e291-47fc-bbc7-ddaebf97612c','6ae5d3b1-4ee3-46e1-891c-3cef020a3f97','05c0bb88-dae5-45f5-9924-1bf40b1fe167','c2c69ab0-1180-46d0-a92d-ea8ef985f904','939513e7-a421-4099-be1a-4c10eb243937','be0a0b1c-50a1-49ec-bbfc-531c21dc44f8','9526c416-185c-4988-8a9c-15a61a4c09af','75bbf0f7-c6bd-479d-863a-b7dc4bdcdf63','07c636b4-d6e8-4c3a-8a54-51b914b93aa5','13761882-8ca3-4c19-80c1-9e609e51b1cf','ee1643ba-a295-4824-aa8c-943b8ce047cc','96d68232-2cd8-4f80-bf9b-e9527a45c944','9e9beea3-8cd5-4e35-9d12-eaa0dde8023c','c56098c1-1f15-4419-ab70-5e9d1af7dc77','4f38d1e0-066e-4489-a365-1b3684ee68af','83f863b7-bdea-4e6e-a629-a49be60961f6','5e945d2f-31a7-4c51-9771-734a12fe41f4','b3ab8f1a-57d5-4358-af3a-b41b9251109c','7d508a2f-29ad-43b7-a7af-e88bde1d0f9d','ac1aa461-7594-4195-87e0-f52afbae92ca','18dce091-1286-4f25-9d65-6e01cb794578','9d43754e-5975-461e-819d-927f96e246fd','901aca05-46f4-42ac-8d30-0cbc2ccd1ecf','4b36812c-ceb7-498d-8b09-75892bbd304a','f70d9bb8-4213-434f-a2d0-77d87ba59d62','132efdc1-7b2a-469f-803f-63b60f57fdd0','ea08b0f5-8d6f-41cd-a66e-4fefa261c710','b3c32e8d-9b42-44ec-bbd6-6d31ed3a8024')), candidates AS MATERIALIZED (SELECT DISTINCT ON(kc.revision_id) kc.revision_id,kc.source_fragment_id,kc.processing_run_id FROM knowledge_candidates kc JOIN sample s ON s.id=kc.revision_id ORDER BY kc.revision_id,kc.created_at DESC)
SELECT json_agg(row_to_json(t)) FROM (SELECT s.public_ref,c.source_fragment_id,c.processing_run_id,sf.content,sf.source_locator FROM sample s LEFT JOIN candidates c ON c.revision_id=s.id LEFT JOIN source_fragments sf ON sf.id=c.source_fragment_id) t;
COMMIT;

```